# Static uniform-sphere gravity workflow

This example builds one analytic model and three immutable midpoint sources for the same sphere. Every setup uses the same two targets. The displayed differences describe these spacings and targets; no new error threshold is implied. The project's separate convergence check remains the scientific acceptance condition.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
from io import BytesIO
import os
import sys
from pathlib import Path

import astropy
import gwexpy
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
from IPython.display import Image, display

from gragra.analytic.objects import uniform_sphere
from gragra.geometry import Frame, Sphere
from gragra.gravity_sources import discretize, point_sum
from gragra.results import compare
from gragra.targets import TargetPoints

frame = Frame("world")
sphere = Sphere(center=[0, 0, 0] * u.m, radius=5 * u.m, frame=frame)
density = 2.7 * u.g / u.cm**3
targets = TargetPoints(
    positions=[[0, 0, 20], [12, 9, 20]] * u.m,
    ids=("A", "B"),
    frame=frame,
)
analytic_model = uniform_sphere(sphere, density)
analytic_setup = analytic_model.at(targets)
spacings = (1.0, 0.5, 0.25)
sources = {h: discretize(sphere, density, h * u.m) for h in spacings}
setups = {h: point_sum(source).at(targets) for h, source in sources.items()}

try:
    gwexpy_version = version("gwexpy")
except PackageNotFoundError:
    gwexpy_version = getattr(gwexpy, "__version__", "unknown")
provenance = {
    "python_executable": str(Path(sys.executable).resolve()),
    "python_version": sys.version.split()[0],
    "python_prefix": str(Path(sys.prefix).resolve()),
    "gragra_import_path": str(Path(__import__("gragra").__file__).resolve().parent),
    "gragra_version": version("gragra"),
    "astropy_version": astropy.__version__,
    "gwexpy_version": gwexpy_version,
    "matplotlib_version": matplotlib.__version__,
    "numpy_version": np.__version__,
}
display({'application/json': provenance}, raw=True)

In [ ]:
# Validate each fixed request before calculation. This checks geometry, target
# regions, and every source-target pair without rebuilding any source.
analytic_validation = analytic_setup.validate()
discrete_validations = {h: setup.validate() for h, setup in setups.items()}
print("analytic:", analytic_validation)
for h, report in discrete_validations.items():
    print(f"h={h:g} m:", report)

In [ ]:
# This picture uses the exact h=0.5 m DiscreteSource retained by its setup.
plot_setup = setups[0.5].plot_setup(view="3d", unit="m", max_sources=250)
plot_figure = plot_setup
plot_png = BytesIO()
plot_figure.savefig(plot_png, format="png", dpi=150, bbox_inches="tight")
plot_bytes = plot_png.getvalue()
plot_path_setting = os.environ.get("GRAGRA_WORKFLOW_PLOT")
if plot_path_setting:
    plot_path = Path(plot_path_setting).expanduser()
    plot_path.parent.mkdir(parents=True, exist_ok=True)
    plot_path.write_bytes(plot_bytes)
plt.close(plot_figure)
display(Image(data=plot_bytes, format="png"))

full_source_positions_m = sources[0.5].positions_m
full_source_count = len(sources[0.5])
displayed_source_count = min(full_source_count, 250)
print("Frozen h=0.5 m source count:", full_source_count)
print("Displayed source count:", displayed_source_count)
print(
    "Full source bounds (m):",
    np.min(full_source_positions_m, axis=0),
    "to",
    np.max(full_source_positions_m, axis=0),
)

## Visual check before interpreting the plot

- [ ] The ideal sphere is centered at (0, 0, 0) m and has radius 5 m.
- [ ] Target A is at (0, 0, 20) m and target B is at (12, 9, 20) m; both IDs are visible.
- [ ] All three axes are labeled in meters and use the world frame.
- [ ] The full h=0.5 m source has 4,224 points and bounds from [-4.75, -4.75, -4.75] m to [4.75, 4.75, 4.75] m, within the ideal sphere extent [-5, -5, -5] m to [5, 5, 5] m.
- [ ] The legend reports 250 displayed points out of all 4,224 frozen sources; confirm the displayed subset is deterministic.

In [ ]:
# Execute the analytic case and the already-discretized cases.
analytic_result = analytic_setup.run()
discrete_results = {h: setup.run() for h, setup in setups.items()}

analytic_evaluation = analytic_result.evaluation
analytic_context = analytic_result.context
assert analytic_context is analytic_evaluation.context
print("shared static context:", analytic_context.frame.name, analytic_context.representation)
print("potential binding shares context:", analytic_context.binding("potential").context is analytic_context)
print("analytic quantities:")
print("potential:", analytic_result.potential)
print("acceleration:", analytic_result.acceleration)
print("gradient:", analytic_result.gradient)
print("analytic parameters:")
print('\n'.join(analytic_evaluation.parameters.pformat(max_lines=-1, max_width=-1)))
print("analytic formulas:")
print(analytic_evaluation.formulas.to_string(index=False))
print("analytic references:")
print(analytic_evaluation.references.to_string(index=False))
print("analytic record:", analytic_result.record)

continuous_mass = sphere.volume * density.to(u.kg / u.m**3)
mass_rows = []
for h, source in sources.items():
    discrete_mass = source.mass.to(u.kg)
    mass_rows.append((h * u.m, source.mass, discrete_mass - continuous_mass))
    print(f"h={h:g} m source bounds (m):", np.min(source.positions_m, axis=0), np.max(source.positions_m, axis=0))
    print(f"h={h:g} m source count:", len(source), "; total mass:", source.mass)
print("continuous mass:", continuous_mass.to(u.kg))
print("spacing, discrete mass, discrete - continuous mass:")
for row in mass_rows:
    print(*row)
discrete_half = discrete_results[0.5]
print("h=0.5 m discrete quantities:")
print("potential:", discrete_half.potential)
print("acceleration:", discrete_half.acceleration)
print("gradient:", discrete_half.gradient)
print("h=0.5 m parameters:")
discrete_evaluation = discrete_half.evaluation
assert discrete_half.context is discrete_evaluation.context
print('\n'.join(discrete_evaluation.parameters.pformat(max_lines=-1, max_width=-1)))
print("h=0.5 m formulas:")
print(discrete_evaluation.formulas.to_string(index=False))
print("h=0.5 m references:")
print(discrete_evaluation.references.to_string(index=False))
print("discrete evaluation record (h=0.5 m):", discrete_half.record)

In [ ]:
# compare() aligns target IDs and positions, then reports candidate-reference
# absolute and relative differences for potential, acceleration, and gradient.
comparisons = {
    h: compare(reference=analytic_result, candidate=result)
    for h, result in discrete_results.items()
}
for h, comparison in comparisons.items():
    print(f"\nspacing h={h:g} m: candidate minus analytic")
    print("potential:", comparison.differences.potential)
    print("acceleration:", comparison.differences.acceleration)
    print("gradient:", comparison.differences.gradient)
    print('\n'.join(comparison.summary.pformat(max_lines=-1, max_width=-1)))
    print("metadata:", comparison.summary.meta)